# EXACT 2026 — Full paper experiments on Google Colab T4

Notebook này thực hiện toàn bộ quy trình từ đầu đến cuối:

1. kiểm tra GPU T4;
2. clone/pull repository từ GitHub và checkout runner đã kiểm định;
3. mount Google Drive;
4. upload hai log chính thức Round 1/2 mà không đưa câu hỏi ẩn lên Git;
5. cài dependency dành riêng cho paper;
6. tải và khóa revision của `Qwen/Qwen2.5-7B-Instruct`;
7. chạy self-test, dry-run, smoke test;
8. chạy full ablation ba lần, telemetry, latency và sinh artifact cho paper.

> Full protocol gồm **9.272 lượt accuracy + 400 lượt latency**. Trên Colab miễn phí, công việc có thể kéo dài qua nhiều phiên. Mọi checkpoint được lưu trên Drive; chỉ cần chạy lại các cell thiết lập rồi chạy lại cell **Full rerun** để resume.

> T4 có khoảng 15 GB VRAM, vì vậy notebook bắt buộc dùng **4-bit NF4**. Đây là điều kiện quantized/non-production-parity và runner sẽ tự ghi nhãn trong báo cáo.

## 0. Chọn GPU T4

Trong Colab chọn **Runtime → Change runtime type → T4 GPU**, sau đó chạy cell dưới đây. Không tiếp tục nếu `CUDA available` là `False`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

subprocess.run(["nvidia-smi"], check=False)

import torch

assert torch.cuda.is_available(), (
    "Không tìm thấy CUDA. Hãy chọn Runtime > Change runtime type > T4 GPU."
)
GPU_NAME = torch.cuda.get_device_name(0)
GPU_GIB = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {GPU_NAME} ({GPU_GIB:.2f} GiB)")
assert "T4" in GPU_NAME.upper(), (
    f"Protocol notebook yêu cầu T4 nhưng runtime hiện tại là {GPU_NAME}. "
    "Hãy chọn Runtime > Change runtime type > T4 GPU."
)


## 1. Clone/pull code từ GitHub

Cell này luôn fetch/pull `main`, sau đó mặc định checkout commit runner đã được kiểm định. Việc pin commit giúp các lần resume không vô tình trộn code mới vào cùng một thí nghiệm.

- Giữ `USE_VALIDATED_COMMIT = True` để tái lập đúng runner đã kiểm thử.
- Chỉ đặt thành `False` nếu bạn chủ động muốn chạy phiên bản mới nhất trên `main`; thay đổi code sẽ tạo experiment directory mới.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/AIVIETNAM-AIO-Triet-Descartes/EXACT2026-NeuroSymbolic-QA.git"
REPO_DIR = Path("/content/EXACT2026-NeuroSymbolic-QA")
BRANCH = "main"
VALIDATED_RUNNER_COMMIT = "f6194ec20af63cc4e6126713c6094d89624b257e"
USE_VALIDATED_COMMIT = True
HIDDEN_LOG_NAMES = (
    "exact_eval_round1_Cay_Nha_La_Vuon.json",
    "exact_eval_round2_Cay_Nha_La_Vuon.json",
)

def exclude_hidden_logs_from_git():
    """Add local-only safeguards even when the pinned commit predates .gitignore updates."""
    exclude_path = REPO_DIR / ".git" / "info" / "exclude"
    existing = exclude_path.read_text(encoding="utf-8") if exclude_path.exists() else ""
    missing = [name for name in HIDDEN_LOG_NAMES if name not in existing.splitlines()]
    if missing:
        prefix = "" if not existing or existing.endswith("\n") else "\n"
        exclude_path.write_text(
            existing + prefix + "\n".join(missing) + "\n",
            encoding="utf-8",
        )

if not (REPO_DIR / ".git").exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    exclude_hidden_logs_from_git()
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH],
        check=True,
    )

exclude_hidden_logs_from_git()

if USE_VALIDATED_COMMIT:
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "checkout", "--detach", VALIDATED_RUNNER_COMMIT],
        check=True,
    )

os.chdir(REPO_DIR)
CODE_COMMIT = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], text=True
).strip()
print(f"Repository: {REPO_DIR}")
print(f"Code commit: {CODE_COMMIT}")
subprocess.run(["git", "status", "--short"], check=True)


## 2. Mount Drive và cung cấp log chính thức

Hai log organizer không được commit lên GitHub vì chứa record của hidden round. Cell upload sẽ lưu chúng vào:

`MyDrive/EXACT2026-paper-inputs/`

Nếu file đã tồn tại trên Drive, notebook sẽ tái sử dụng và không hỏi upload lại. Checkpoint/kết quả luôn nằm trên Drive.

Model Qwen cần khoảng 15 GB checkpoint gốc. Cache model mặc định nằm ở ổ tạm `/content` để không làm đầy Drive miễn phí 15 GB. Nếu Drive có **ít nhất 20 GiB trống** và bạn muốn tái sử dụng model qua nhiều phiên, đặt `PERSIST_MODEL_CACHE_ON_DRIVE = True`.

> Giữ `EXACT2026-paper-inputs` ở chế độ private: hai JSON là log hidden round dạng plaintext. Chỉ chia sẻ thư mục kết quả đã tổng hợp.

In [ ]:
from google.colab import drive
import os
import shutil
from pathlib import Path

drive.mount("/content/drive", force_remount=False)

MY_DRIVE = Path("/content/drive/MyDrive")
INPUT_DIR = MY_DRIVE / "EXACT2026-paper-inputs"
RESULTS_ROOT = MY_DRIVE / "EXACT2026-paper-results"
PERSIST_MODEL_CACHE_ON_DRIVE = False
HF_CACHE_ROOT = (
    MY_DRIVE / "EXACT2026-huggingface-cache"
    if PERSIST_MODEL_CACHE_ON_DRIVE
    else Path("/content/EXACT2026-huggingface-cache")
)
HF_HUB_CACHE = HF_CACHE_ROOT / "hub"
PINNED_QWEN_COMMIT = "a09a35458c702b33eeacc393d103063234e8bc28"
PINNED_EMBEDDING_COMMIT = "1110a243fdf4706b3f48f1d95db1a4f5529b4d41"
for directory in (INPUT_DIR, RESULTS_ROOT, HF_CACHE_ROOT, HF_HUB_CACHE):
    directory.mkdir(parents=True, exist_ok=True)

qwen_snapshot = (
    HF_HUB_CACHE / "models--Qwen--Qwen2.5-7B-Instruct" /
    "snapshots" / PINNED_QWEN_COMMIT
)
encoder_snapshot = (
    HF_HUB_CACHE / "models--sentence-transformers--all-MiniLM-L6-v2" /
    "snapshots" / PINNED_EMBEDDING_COMMIT
)
MODEL_CACHE_READY = all(
    path.exists() for path in (
        qwen_snapshot / "model.safetensors.index.json",
        qwen_snapshot / "model-00001-of-00004.safetensors",
        qwen_snapshot / "model-00002-of-00004.safetensors",
        qwen_snapshot / "model-00003-of-00004.safetensors",
        qwen_snapshot / "model-00004-of-00004.safetensors",
        encoder_snapshot / "model.safetensors",
    )
)
cache_free_gib = shutil.disk_usage(HF_CACHE_ROOT).free / 1024**3
if not MODEL_CACHE_READY and cache_free_gib < 20:
    raise RuntimeError(
        f"Ổ cache chỉ còn {cache_free_gib:.1f} GiB; cần ít nhất 20 GiB để tải model. "
        "Hãy giải phóng dung lượng hoặc đổi PERSIST_MODEL_CACHE_ON_DRIVE."
    )

os.environ["PAPER_OUTPUT_DIR"] = str(RESULTS_ROOT)
os.environ["HF_HOME"] = str(HF_CACHE_ROOT)
os.environ["HF_HUB_CACHE"] = str(HF_HUB_CACHE)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print(f"Input logs: {INPUT_DIR}")
print(f"Experiment outputs: {RESULTS_ROOT}")
print(f"Persistent HF cache root: {HF_CACHE_ROOT}")
print(f"Model snapshot cache: {HF_HUB_CACHE}")
print(f"Cache free space: {cache_free_gib:.1f} GiB")
print(f"Pinned snapshots already complete: {MODEL_CACHE_READY}")


In [ ]:
from google.colab import files
from pathlib import Path
import hashlib
import json

ROUND1_NAME = "exact_eval_round1_Cay_Nha_La_Vuon.json"
ROUND2_NAME = "exact_eval_round2_Cay_Nha_La_Vuon.json"
EXPECTED_LOG_NAMES = (ROUND1_NAME, ROUND2_NAME)
EXPECTED_LOG_IDENTITY = {
    ROUND1_NAME: {
        "sha256": "6d5e7a86a5e0a7ed1e1c3e9f43b7228bd930d0e4a7a6133f62ad302483b7fd4b",
        "eval_round": "round1",
        "sample_version": "eval-round1-type1-type2-v2",
        "score": 39.38,
    },
    ROUND2_NAME: {
        "sha256": "03032ec92f384d3c0ccf76e9f7801cf0b107452805b97115b00061e9c6fdc813",
        "eval_round": "round2",
        "sample_version": "eval-round2-type1-type2-v3",
        "score": 44.8,
    },
}

missing = [name for name in EXPECTED_LOG_NAMES if not (INPUT_DIR / name).exists()]
if missing:
    print("Hãy chọn các file còn thiếu:", missing)
    uploaded = files.upload()
    for name in missing:
        if name not in uploaded:
            raise FileNotFoundError(f"Chưa upload đúng file: {name}")
        (INPUT_DIR / name).write_bytes(uploaded[name])

ROUND1_LOG = INPUT_DIR / ROUND1_NAME
ROUND2_LOG = INPUT_DIR / ROUND2_NAME

for path in (ROUND1_LOG, ROUND2_LOG):
    payload = json.loads(path.read_text(encoding="utf-8"))
    expected = EXPECTED_LOG_IDENTITY[path.name]
    assert isinstance(payload.get("logs"), list), f"{path.name}: thiếu logs[]"
    assert isinstance(payload.get("summary"), dict), f"{path.name}: thiếu summary"
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    assert digest == expected["sha256"], f"{path.name}: SHA-256 không đúng log đã cấp"
    assert payload.get("eval_round") == expected["eval_round"], path.name
    assert payload.get("sample_version") == expected["sample_version"], path.name
    assert len(payload["logs"]) == 50, f"{path.name}: expected 50 records"
    assert payload["summary"].get("team") == "Cây Nhà Lá Vườn", path.name
    assert float(payload["summary"].get("total_points")) == expected["score"], path.name
    print(
        f"OK {path.name}: n={len(payload['logs'])}, "
        f"sample={payload.get('sample_version')}, sha256={digest[:16]}..."
    )

# Self-test hiện dùng hai tên chuẩn tại repo root. Symlink không sao chép dữ liệu
# hidden vào Git và biến mất cùng runtime Colab.
for source in (ROUND1_LOG, ROUND2_LOG):
    target = REPO_DIR / source.name
    if target.is_symlink():
        target.unlink()
    elif target.exists():
        if hashlib.sha256(target.read_bytes()).hexdigest() != hashlib.sha256(source.read_bytes()).hexdigest():
            raise RuntimeError(f"File khác nội dung đã tồn tại: {target}")
        continue
    target.symlink_to(source)

print("Official logs ready. Không in nội dung hidden queries.")


## 3. Cài dependency cho T4

Không cài `vllm` và không thay Torch/CUDA có sẵn của Colab. `transformers<5` tránh thay đổi API lớn ngoài protocol đã kiểm thử. Lần đầu notebook khóa cả package trực tiếp lẫn các dependency ML quan trọng (`huggingface-hub`, tokenizer, safetensors, SciPy, scikit-learn). Các phiên resume cài lại đúng lock, chạy `pip check`, và đưa toàn bộ version cùng Torch/CUDA vào fingerprint của experiment.

In [ ]:
import hashlib
import importlib.metadata
import json
import subprocess
import sys

PAPER_PACKAGE_SPECS = {
    "fastapi": "fastapi>=0.110",
    "pydantic": "pydantic>=2",
    "httpx": "httpx>=0.27",
    "pyyaml": "pyyaml>=6",
    "loguru": "loguru>=0.7",
    "openai": "openai>=1.30",
    "z3-solver": "z3-solver>=4.13",
    "sympy": "sympy>=1.12",
    "numpy": "numpy>=1.24,<3",
    "matplotlib": "matplotlib>=3.7",
    "transformers": "transformers>=4.46,<5",
    "accelerate": "accelerate>=0.30,<2",
    "bitsandbytes": "bitsandbytes>=0.43,<1",
    "faiss-cpu": "faiss-cpu>=1.8",
    "sentence-transformers": "sentence-transformers>=2.7,<6",
}
RUNTIME_LOCK_PACKAGES = tuple(dict.fromkeys((
    *PAPER_PACKAGE_SPECS.keys(),
    "huggingface-hub",
    "tokenizers",
    "safetensors",
    "scipy",
    "scikit-learn",
)))
DEPENDENCY_LOCK = INPUT_DIR / "paper-dependencies.lock.json"
if DEPENDENCY_LOCK.exists():
    locked_versions = json.loads(DEPENDENCY_LOCK.read_text(encoding="utf-8"))
    missing_from_lock = sorted(set(RUNTIME_LOCK_PACKAGES) - set(locked_versions))
    if missing_from_lock:
        raise RuntimeError(
            "Dependency lock cũ/chưa đầy đủ: " + ", ".join(missing_from_lock) +
            ". Đổi tên lock cũ để tạo experiment environment mới."
        )
    INSTALL_SPECS = [
        f"{name}=={locked_versions[name]}" for name in RUNTIME_LOCK_PACKAGES
    ]
    print(f"Using dependency lock: {DEPENDENCY_LOCK}")
else:
    locked_versions = {}
    INSTALL_SPECS = list(PAPER_PACKAGE_SPECS.values())

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", *INSTALL_SPECS],
    check=True,
)

RESOLVED_PAPER_VERSIONS = {
    name: importlib.metadata.version(name) for name in RUNTIME_LOCK_PACKAGES
}
if DEPENDENCY_LOCK.exists():
    mismatched = {
        name: {"locked": locked_versions[name], "installed": version}
        for name, version in RESOLVED_PAPER_VERSIONS.items()
        if locked_versions[name] != version
    }
    if mismatched:
        raise RuntimeError(f"Installed versions differ from lock: {mismatched}")
else:
    DEPENDENCY_LOCK.write_text(
        json.dumps(RESOLVED_PAPER_VERSIONS, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )
    print(f"Created dependency lock: {DEPENDENCY_LOCK}")

pip_check = subprocess.run(
    [sys.executable, "-m", "pip", "check"],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(pip_check.stdout.strip())
PIP_CHECK_LOG = INPUT_DIR / "paper-pip-check.txt"
PIP_CHECK_LOG.write_text(pip_check.stdout, encoding="utf-8")
if pip_check.returncode != 0:
    print(
        "WARNING: pip check phát hiện conflict trong image Colab. Kết quả sẽ được "
        "fingerprint; nếu smoke fail, hãy restart runtime và xem paper-pip-check.txt."
    )

TRACKED_PACKAGES = ("torch", *RUNTIME_LOCK_PACKAGES)
PACKAGE_VERSIONS = {}
for package in TRACKED_PACKAGES:
    try:
        PACKAGE_VERSIONS[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        PACKAGE_VERSIONS[package] = None

RUNTIME_ENVIRONMENT = {
    "python": sys.version.split()[0],
    "cuda": torch.version.cuda,
    "gpu": GPU_NAME,
    "packages": PACKAGE_VERSIONS,
    "pip_check_returncode": pip_check.returncode,
    "pip_check_sha256": hashlib.sha256(pip_check.stdout.encode("utf-8")).hexdigest(),
}
ENV_FINGERPRINT = hashlib.sha256(
    json.dumps(RUNTIME_ENVIRONMENT, sort_keys=True).encode("utf-8")
).hexdigest()[:10]
RUNTIME_MANIFEST_PATH = INPUT_DIR / f"colab-runtime-{ENV_FINGERPRINT}.json"
RUNTIME_MANIFEST_PATH.write_text(
    json.dumps(RUNTIME_ENVIRONMENT, indent=2) + "\n", encoding="utf-8"
)
print(json.dumps(RUNTIME_ENVIRONMENT, indent=2))
print(f"Runtime fingerprint: {ENV_FINGERPRINT}")


## 4. Tải Qwen và semantic encoder đã khóa revision

Hai model đều public nên không cần Hugging Face token. Notebook dùng commit SHA bất biến đã ghi trong protocol, tải Qwen cùng encoder khớp với FAISS index, rồi lưu identity vào Drive. Cache nằm ở vị trí đã chọn tại cell 2.

4-bit chỉ giảm VRAM khi **load model**; vẫn cần tải checkpoint safetensors gốc.

In [ ]:
import os
import re
from pathlib import Path

# Hai biến cache phải được đặt trước khi import huggingface_hub/transformers.
os.environ["HF_HOME"] = str(HF_CACHE_ROOT)
os.environ["HF_HUB_CACHE"] = str(HF_HUB_CACHE)

from huggingface_hub import snapshot_download

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
MODEL_COMMIT = PINNED_QWEN_COMMIT
EMBEDDING_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_MODEL_COMMIT = PINNED_EMBEDDING_COMMIT
MODEL_COMMIT_FILE = INPUT_DIR / "qwen2.5-7b-instruct.commit.txt"
EMBEDDING_COMMIT_FILE = INPUT_DIR / "all-minilm-l6-v2.commit.txt"

for identity_file, expected_commit in (
    (MODEL_COMMIT_FILE, MODEL_COMMIT),
    (EMBEDDING_COMMIT_FILE, EMBEDDING_MODEL_COMMIT),
):
    if identity_file.exists():
        observed = identity_file.read_text(encoding="utf-8").strip()
        if observed != expected_commit:
            raise RuntimeError(
                f"{identity_file.name} chứa commit {observed}, khác protocol {expected_commit}. "
                "Hãy dùng Drive/input folder riêng cho protocol này."
            )
    else:
        identity_file.write_text(expected_commit + "\n", encoding="utf-8")

assert re.fullmatch(r"[0-9a-fA-F]{40}", MODEL_COMMIT), MODEL_COMMIT
assert re.fullmatch(r"[0-9a-fA-F]{40}", EMBEDDING_MODEL_COMMIT)
print(f"Pinned model: {MODEL_ID}@{MODEL_COMMIT}")
print(f"Pinned encoder: {EMBEDDING_MODEL_ID}@{EMBEDDING_MODEL_COMMIT}")

MODEL_SNAPSHOT = Path(
    snapshot_download(
        repo_id=MODEL_ID,
        revision=MODEL_COMMIT,
        cache_dir=str(HF_HUB_CACHE),
        ignore_patterns=["*.bin", "*.msgpack", "*.h5", "*.ot", "original/*"],
    )
)
snapshot_bytes = sum(
    path.stat().st_size for path in MODEL_SNAPSHOT.rglob("*") if path.is_file()
)
print(f"Snapshot ready: {MODEL_SNAPSHOT}")
print(f"Logical snapshot size: {snapshot_bytes / 1024**3:.2f} GiB")

EMBEDDING_SNAPSHOT = Path(
    snapshot_download(
        repo_id=EMBEDDING_MODEL_ID,
        revision=EMBEDDING_MODEL_COMMIT,
        cache_dir=str(HF_HUB_CACHE),
        ignore_patterns=[
            "onnx/*", "openvino/*", "*.bin", "*.h5", "*.ot", "*.msgpack"
        ],
    )
)
print(f"Encoder snapshot ready: {EMBEDDING_SNAPSHOT}")

# One-time provenance check: the checked-in FAISS vectors must equal vectors
# regenerated with the pinned encoder and the exact index-build text template.
import gc
import pickle
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

index_dir = REPO_DIR / "data" / "formula_index"
encoder_manifest = json.loads(
    (index_dir / "encoder.json").read_text(encoding="utf-8")
)
assert encoder_manifest["model"] == EMBEDDING_MODEL_ID
assert encoder_manifest["revision"] == EMBEDDING_MODEL_COMMIT
formula_index = faiss.read_index(str(index_dir / "index.faiss"))
with (index_dir / "metadata.pkl").open("rb") as handle:
    formula_docs = pickle.load(handle)
index_texts = [
    f"{doc['domain']}: {doc['formula_natural']} — {' '.join(doc.get('keywords', []))}"
    for doc in formula_docs
]
embedding_verifier = SentenceTransformer(
    EMBEDDING_MODEL_ID,
    revision=EMBEDDING_MODEL_COMMIT,
    cache_folder=str(HF_HUB_CACHE),
    device="cpu",
)
fresh_vectors = embedding_verifier.encode(
    index_texts, show_progress_bar=False
).astype("float32")
stored_vectors = np.vstack([
    formula_index.reconstruct(i) for i in range(formula_index.ntotal)
]).astype("float32")
np.testing.assert_allclose(stored_vectors, fresh_vectors, rtol=1e-5, atol=1e-5)
print(
    "FAISS provenance PASS: "
    f"{formula_index.ntotal} vectors, max |delta|="
    f"{np.max(np.abs(stored_vectors - fresh_vectors)):.3e}"
)
del embedding_verifier, fresh_vectors, stored_vectors
gc.collect()


## 5. Self-test và dry-run

Hai bước này không chạy full inference:

- `self-test`: kiểm tra loader, official scorer, Z3/PAL executor, unit conversion, cache và resume;
- `dry-run`: tái tạo điểm chính thức, latency organizer, work estimate và toàn bộ report schema.

In [ ]:
import subprocess
import sys

SELF_TEST_CMD = [
    sys.executable,
    "paper/run_paper_experiments.py",
    "--mode", "self-test",
    "--install-deps", "no",
]
VALIDATION_DIR = RESULTS_ROOT / "notebook_validation"
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)
SELF_TEST_LOG = VALIDATION_DIR / f"self_test_{CODE_COMMIT[:8]}_{ENV_FINGERPRINT}.log"
self_test = subprocess.run(
    SELF_TEST_CMD,
    cwd=REPO_DIR,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
SELF_TEST_LOG.write_text(self_test.stdout, encoding="utf-8")
print(self_test.stdout)
print(f"Durable self-test log: {SELF_TEST_LOG}")
self_test.check_returncode()


In [ ]:
DRY_RUN_NAME = "colab_t4_dry_run"
DRY_RUN_CMD = [
    sys.executable,
    "paper/run_paper_experiments.py",
    "--mode", "dry-run",
    "--embedding-model", EMBEDDING_MODEL_ID,
    "--embedding-model-revision", EMBEDDING_MODEL_COMMIT,
    "--round1-log", str(ROUND1_LOG),
    "--round2-log", str(ROUND2_LOG),
    "--output-dir", str(RESULTS_ROOT),
    "--run-name", DRY_RUN_NAME,
    "--install-deps", "no",
]
subprocess.run(DRY_RUN_CMD, cwd=REPO_DIR, check=True)

def latest_run_dir(run_name):
    candidates = [
        path for path in RESULTS_ROOT.glob(run_name + "*")
        if (path / "run_config.json").exists()
    ]
    if not candidates:
        raise FileNotFoundError(f"Không tìm thấy output cho {run_name}")
    return max(candidates, key=lambda path: (path / "run_config.json").stat().st_mtime)

DRY_RUN_DIR = latest_run_dir(DRY_RUN_NAME)
print(f"Dry-run report: {DRY_RUN_DIR / 'paper_results.md'}")


## 6. Smoke test bằng Qwen thật trên T4

Smoke test chạy 60 logical jobs trên tập con nhỏ, bao gồm accuracy, latency, Z3, PAL, repair và report. Lần đầu cell này cũng load Qwen ở 4-bit và chuẩn bị semantic formula RAG trên CPU.

Đặt `RUN_SMOKE = False` chỉ khi bạn đã chạy thành công smoke test trước đó với cùng code/model.

In [ ]:
RUN_SMOKE = True
SMOKE_RUN_NAME = (
    f"colab_t4_smoke_{MODEL_COMMIT[:8]}_"
    f"{EMBEDDING_MODEL_COMMIT[:8]}_{ENV_FINGERPRINT}"
)

SMOKE_CMD = [
    sys.executable,
    "paper/run_paper_experiments.py",
    "--mode", "smoke",
    "--backend", "transformers",
    "--model", MODEL_ID,
    "--model-revision", MODEL_COMMIT,
    "--embedding-model", EMBEDDING_MODEL_ID,
    "--embedding-model-revision", EMBEDDING_MODEL_COMMIT,
    "--quantization", "4bit",
    "--round1-log", str(ROUND1_LOG),
    "--round2-log", str(ROUND2_LOG),
    "--output-dir", str(RESULTS_ROOT),
    "--run-name", SMOKE_RUN_NAME,
    "--install-deps", "no",
    "--progress-every", "1",
]

if RUN_SMOKE:
    subprocess.run(SMOKE_CMD, cwd=REPO_DIR, check=True, env=os.environ.copy())
    SMOKE_RUN_DIR = latest_run_dir(SMOKE_RUN_NAME)
    smoke_quality = json.loads(
        (SMOKE_RUN_DIR / "metrics" / "quality_gate.json").read_text(encoding="utf-8")
    )
    smoke_config = json.loads(
        (SMOKE_RUN_DIR / "run_config.json").read_text(encoding="utf-8")
    )
    assert smoke_quality["complete_for_requested_subset"] is True
    assert smoke_quality["expected_total"] == smoke_quality["completed_total"] == 60
    assert smoke_quality["failed_total"] == 0
    assert smoke_quality["infrastructure_failed_total"] == 0
    assert smoke_config["semantic_rag"]["status"] == "loaded"
    assert smoke_config["resolved"]["resolved_model_revision"] == MODEL_COMMIT
    assert (
        smoke_config["resolved"]["resolved_embedding_model_revision"]
        == EMBEDDING_MODEL_COMMIT
    )
    print("Smoke quality gate: PASS (60/60, no failures).")
    print(f"Smoke report: {SMOKE_RUN_DIR / 'paper_results.md'}")
else:
    print("Smoke test skipped by user.")


## 7. Full rerun — chạy cell này và chạy lại để resume

Protocol cố định:

- toàn bộ 808 Type-1 + 200 Type-2 public examples;
- ba repeat cho mọi cấu hình có LLM;
- một repeat cho deterministic `t2_rag_solver`;
- 50 mẫu/variant cho latency không cache, không retry;
- semantic RAG, Z3, PAL, self-repair và paired bootstrap;
- checkpoint append-only trên Drive.

Nếu Colab ngắt kết nối: kết nối lại T4, chạy lại từ cell 0 đến cell tải model, có thể bỏ smoke, rồi chạy lại chính cell Full rerun này. **Không đổi `FULL_RUN_NAME`, model commit, code commit hoặc tham số** nếu muốn resume cùng experiment.

In [ ]:
assert USE_VALIDATED_COMMIT is True, "Full protocol phải dùng validated commit"
assert CODE_COMMIT == VALIDATED_RUNNER_COMMIT

FULL_RUN_NAME = (
    f"qwen25_7b_t4_full_{MODEL_COMMIT[:8]}_"
    f"{EMBEDDING_MODEL_COMMIT[:8]}_{ENV_FINGERPRINT}"
)
FULL_RUN_DIR = RESULTS_ROOT / FULL_RUN_NAME

FULL_CMD = [
    sys.executable,
    "paper/run_paper_experiments.py",
    "--mode", "full",
    "--tracks", "both",
    "--backend", "transformers",
    "--model", MODEL_ID,
    "--model-revision", MODEL_COMMIT,
    "--embedding-model", EMBEDDING_MODEL_ID,
    "--embedding-model-revision", EMBEDDING_MODEL_COMMIT,
    "--quantization", "4bit",
    "--temperature", "0.1",
    "--max-tokens", "1024",
    "--llm-timeout", "120",
    "--code-timeout", "8",
    "--repeats", "3",
    "--deterministic-repeats", "1",
    "--seed", "2026",
    "--latency-samples", "50",
    "--bootstrap-samples", "2000",
    "--max-retries", "2",
    "--cache-shared-stages",
    "--resume",
    "--round1-log", str(ROUND1_LOG),
    "--round2-log", str(ROUND2_LOG),
    "--output-dir", str(RESULTS_ROOT),
    "--run-name", FULL_RUN_NAME,
    "--install-deps", "no",
    "--progress-every", "25",
]

print("Running:", " ".join(FULL_CMD))
result = subprocess.run(FULL_CMD, cwd=REPO_DIR, env=os.environ.copy())
print(f"Runner exit code: {result.returncode}")
if result.returncode == 0:
    print("Full protocol complete and quality gate passed.")
elif result.returncode in (2, 130):
    print(
        "Run đã checkpoint nhưng chưa paper-ready. Chạy lại cùng cell để resume; "
        "xem metrics/quality_gate.json nếu có job lỗi."
    )
else:
    raise RuntimeError(f"Runner failed with exit code {result.returncode}")


## 8. Xem tiến độ và kết quả

Runner có thể tạo suffix theo config hash nếu code/GPU/config thay đổi. Cell dưới đây tìm experiment directory mới nhất khớp `FULL_RUN_NAME`, đọc quality gate và hiển thị report.

In [ ]:
import json
from pathlib import Path
from IPython.display import Markdown, display

candidates = [
    path
    for path in RESULTS_ROOT.glob(FULL_RUN_NAME + "*")
    if (path / "run_config.json").exists()
]
if not candidates:
    raise FileNotFoundError(f"Chưa tìm thấy run cho {FULL_RUN_NAME}")

ACTIVE_RUN_DIR = max(
    candidates,
    key=lambda path: (path / "run_config.json").stat().st_mtime,
)
print(f"Active run: {ACTIVE_RUN_DIR}")

run_config = json.loads(
    (ACTIVE_RUN_DIR / "run_config.json").read_text(encoding="utf-8")
)
environment = json.loads(
    (ACTIVE_RUN_DIR / "environment.json").read_text(encoding="utf-8")
)
repro_dir = ACTIVE_RUN_DIR / "reproducibility"
repro_dir.mkdir(exist_ok=True)
for source in (
    DEPENDENCY_LOCK,
    RUNTIME_MANIFEST_PATH,
    PIP_CHECK_LOG,
    MODEL_COMMIT_FILE,
    EMBEDDING_COMMIT_FILE,
):
    shutil.copy2(source, repro_dir / source.name)

quality_path = ACTIVE_RUN_DIR / "metrics" / "quality_gate.json"
if quality_path.exists():
    quality = json.loads(quality_path.read_text(encoding="utf-8"))
    keys = (
        "paper_ready",
        "complete_for_requested_subset",
        "expected_total",
        "completed_total",
        "failed_total",
        "infrastructure_failed_total",
    )
    print({key: quality.get(key) for key in keys})
    if quality.get("paper_ready"):
        assert USE_VALIDATED_COMMIT is True
        assert CODE_COMMIT == VALIDATED_RUNNER_COMMIT
        assert VALIDATED_RUNNER_COMMIT == "f6194ec20af63cc4e6126713c6094d89624b257e"
        arguments = run_config["arguments"]
        resolved = run_config["resolved"]
        expected_protocol = {
            "mode": "full",
            "tracks": "both",
            "backend": "transformers",
            "model": MODEL_ID,
            "model_revision": MODEL_COMMIT,
            "embedding_model": EMBEDDING_MODEL_ID,
            "embedding_model_revision": EMBEDDING_MODEL_COMMIT,
            "quantization": "4bit",
            "temperature": 0.1,
            "max_tokens": 1024,
            "llm_timeout": 120.0,
            "code_timeout": 8.0,
            "repeats": 3,
            "deterministic_repeats": 1,
            "seed": 2026,
            "latency_samples": 50,
            "bootstrap_samples": 2000,
            "max_retries": 2,
            "cache_shared_stages": True,
            "disable_semantic_rag": False,
            "type1_limit": 0,
            "type2_limit": 0,
        }
        mismatches = {
            key: {"expected": expected, "observed": arguments.get(key)}
            for key, expected in expected_protocol.items()
            if arguments.get(key) != expected
        }
        assert not mismatches, f"Protocol mismatch: {mismatches}"
        assert resolved["resolved_model_revision"] == MODEL_COMMIT
        assert resolved["resolved_embedding_model_revision"] == EMBEDDING_MODEL_COMMIT
        assert resolved["quantization"] == "4bit"
        assert environment["git"]["commit_sha"] == CODE_COMMIT
        assert environment["git"]["dirty"] is False
        assert "T4" in environment["gpu"]["devices"][0]["name"].upper()
        assert quality["expected_total"] == quality["completed_total"] == 9672
        assert (ACTIVE_RUN_DIR / "PAPER_READY").exists()
        print("EXACT PROTOCOL VERIFIED: T4 + pinned models + full matrix + clean code.")
else:
    print("quality_gate.json chưa được tạo; run có thể vẫn đang ở lần đầu.")

report_path = ACTIVE_RUN_DIR / "paper_results.md"
if report_path.exists():
    display(Markdown(report_path.read_text(encoding="utf-8")))
else:
    print("Report chưa được tạo. Xem log:", ACTIVE_RUN_DIR / "logs" / "runner.log")


In [ ]:
# In nhanh 80 dòng log cuối mà không đọc/hiển thị hidden query content.
log_path = ACTIVE_RUN_DIR / "logs" / "runner.log"
if log_path.exists():
    log_lines = log_path.read_text(encoding="utf-8", errors="replace").splitlines()
    print("\n".join(log_lines[-80:]))
else:
    print("Chưa có runner.log")


## Troubleshooting T4/Colab

- **CUDA out of memory:** kiểm tra lệnh full vẫn có `--quantization 4bit`; restart runtime để giải phóng model cũ rồi chạy lại các cell. Không đổi sang FP16 trên T4.
- **Colab ngắt phiên:** không xóa thư mục Drive. Chạy lại notebook với cùng code/model/dependency lock rồi chạy lại cell Full rerun.
- **Dependency lock không còn tương thích với Python/CUDA mới:** đổi tên hoặc xóa có chủ đích `paper-dependencies.lock.json`, chạy lại cell cài đặt và chấp nhận một `ENV_FINGERPRINT`/experiment directory mới. Không gộp số liệu của hai environment.
- **Semantic RAG không load:** full runner cố ý dừng thay vì âm thầm chuyển condition. Sửa dependency/network; chỉ dùng `--disable-semantic-rag` cho một thí nghiệm keyword-only tách biệt.
- **Drive đầy:** model cache và experiment logs có thể chiếm nhiều GB. Không xóa `predictions.jsonl`, `events.jsonl`, `stage_cache.jsonl` của run đang resume.
- **Exit code 2:** mở `metrics/quality_gate.json` và `errors.jsonl`, sau đó chạy lại cùng lệnh. Failed/infrastructure-failed jobs sẽ được retry.



## Artifact cần dùng cho paper

Khi quality gate tạo file `PAPER_READY`, các artifact quan trọng nằm trong `ACTIVE_RUN_DIR`:

- `paper_results.md`: báo cáo tổng hợp;
- `metrics/summary.json`: toàn bộ metric có cấu trúc;
- `metrics/ablation.csv`: kết quả ablation;
- `metrics/component_stats.csv`: Z3/PAL/self-repair;
- `metrics/latency.csv`: latency controlled;
- `metrics/quality_gate.json`: completeness/failure gate;
- `tables/*.csv|*.md|*.tex`: bảng paper-ready;
- `cases/case_studies.md|json`: 3–5 case study public;
- `figures/architecture.png|pdf|mmd`: kiến trúc hệ thống;
- `predictions.jsonl`, `events.jsonl`, `errors.jsonl`: audit trail đầy đủ.

Không dùng số liệu của smoke/dry-run làm kết quả paper. Không diễn giải chênh lệch Round 1→2 như paired improvement vì hai round dùng sample version/query set khác nhau.